# Multi-agent systems

Assistant version 10: Route to billing, product RAG and logistics SQL specialists; supervise the combined answer.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra deep --group authoring`, then `uv run --extra deep --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## Subagents exposed as tools; supervisor; restricted tool sets

Predict the output before running. The function definitions below match `examples/01_supervisor_tools.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Expose a read-only logistics specialist as a supervisor tool.

Run: uv run --extra agents python modules/16_multi_agent/examples/01_supervisor_tools.py --offline
Expected: The supervisor delegates; the child performs one scoped SQL tool call.
Cost: At most six short calls across supervisor and child. Offline fixtures incur no API fee.
Requires: agents
"""

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware
from langchain.tools import tool

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.operations import run_specialist
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()

    @tool
    def ask_logistics(order_id: str) -> dict:
        """Ask logistics for a read-only shipment report for this authenticated customer."""
        return run_specialist(settings, "C001", "logistics", order_id, "BW-001")

    model = get_chat_model(
        settings=settings,
        responses=[
            tool_request("ask_logistics", {"order_id": "BW1001"}),
            "Logistics evidence attached.",
        ],
    )
    # KEY IDEA: only the wrapper knows the customer; neither supervisor nor child chooses it.
    supervisor = create_agent(
        model,
        [ask_logistics],
        middleware=[
            ModelCallLimitMiddleware(run_limit=3, exit_behavior="end"),
            ToolCallLimitMiddleware(run_limit=1, exit_behavior="end"),
        ],
    )
    state = supervisor.invoke(
        {"messages": [{"role": "user", "content": "Where is order BW1001?"}]},
        {"recursion_limit": 30},
    )
    print_text(str([m.content for m in state["messages"] if m.type == "tool"]))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Command-based handoffs; explicit ownership and return routing

Predict the output before running. The function definitions below match `examples/02_handoffs.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Transfer ownership with Command and explicitly return to the supervisor.

Run: uv run --extra agents python modules/16_multi_agent/examples/02_handoffs.py --offline
Expected: Supervisor hands off to billing, then receives the read-only result.
Cost: At most three short specialist model calls. Offline fixtures incur no API fee.
Requires: agents
"""

from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

from common.errors import run_cli
from common.lesson import lesson_settings
from common.operations import run_specialist
from common.pretty import print_text


class State(TypedDict):
    owner: str
    report: dict


def main() -> None:
    settings = lesson_settings()

    def supervisor(state: State) -> Command[Literal["billing", "__end__"]]:
        return (
            Command(update={"owner": "supervisor"}, goto=END)
            if state["report"]
            else Command(update={"owner": "billing"}, goto="billing")
        )

    def billing(state: State) -> Command[Literal["supervisor"]]:
        report = run_specialist(settings, "C001", "billing", "BW1001", "BW-001")
        return Command(update={"owner": "supervisor", "report": report}, goto="supervisor")

    # KEY IDEA: ownership is explicit state; no separate static edge duplicates the handoff.
    graph = StateGraph(State)
    graph.add_node("supervisor", supervisor)
    graph.add_node("billing", billing)
    graph.add_edge(START, "supervisor")
    print_text(
        str(graph.compile().invoke({"owner": "supervisor", "report": {}}, {"recursion_limit": 5}))
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Specialist router; confidence and fallback handling

Predict the output before running. The function definitions below match `examples/03_router.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Use confidence to select specialists or clarification.

Run: uv run --extra agents python modules/16_multi_agent/examples/03_router.py --offline
Expected: A shipping request selects logistics; an unclear request uses fallback.
Cost: At most seven short calls across two demonstrations. Offline fixtures incur no API fee.
Requires: agents
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.operations import build_operations
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    # KEY IDEA: each invocation starts a new isolated graph run with an empty reducer.
    for question in ("Check shipping for BW1001", "I need help"):
        result = build_operations(settings, "C001").invoke(
            {"question": question, "order_id": "BW1001", "sku": "BW-001", "reports": []},
            {"max_concurrency": 2},
        )
        print_text(str({"roles": result["roles"], "answer": result["answer"]}))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Shared versus private state; context engineering; minimum necessary history

Predict the output before running. The function definitions below match `examples/04_context_boundaries.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Give specialists only the identifiers needed for their jobs.

Run: uv run --extra agents python modules/16_multi_agent/examples/04_context_boundaries.py --offline
Expected: Product sees the SKU; logistics sees the order; neither sees the full history.
Cost: At most three short calls; cross-customer access is denied by SQL. Offline fixtures incur
no API fee.
Requires: agents
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.operations import run_specialist, specialist_input
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    private_history = "Synthetic private customer note: demo@example.test"
    for role in ("product", "logistics"):
        message = specialist_input(role, "BW1001", "BW-001")
        assert private_history not in message
        print_text(f"{role} receives: {message}")
    # KEY IDEA: projection is code; a prompt saying ignore private history is not isolation.
    report = run_specialist(settings, "C002", "logistics", "BW1001", "BW-001")
    print_text(str(report))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## When multiple agents help or hurt; latency, token use and quality comparison

Predict the output before running. The function definitions below match `examples/05_compare_single.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Measure call count, approximate input tokens and latency for one versus two agents.

Run: uv run --extra agents python modules/16_multi_agent/examples/05_compare_single.py --offline
Expected: Both paths expose evidence; delegation adds model work without proving better quality.
Cost: At most nine short calls across both measured paths. Offline fixtures incur no API fee.
Requires: agents
"""

from time import perf_counter

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.operations import specialist, specialist_input
from common.pretty import print_text


class Meter(BaseCallbackHandler):
    def __init__(self) -> None:
        self.calls = 0
        self.characters = 0

    def on_chat_model_start(self, serialized: dict, messages: list, **kwargs: object) -> None:
        self.calls += 1
        self.characters += sum(len(str(m.content)) for batch in messages for m in batch)


def main() -> None:
    settings = lesson_settings()
    child = specialist(settings, "C001", "logistics", "BW1001", "BW-001")

    @tool
    def delegate(order_id: str) -> str:
        """Read shipment facts for the requested order through a specialist."""
        result = child.invoke(
            {
                "messages": [
                    {"role": "user", "content": specialist_input("logistics", order_id, "BW-001")}
                ]
            }
        )
        return str([m.content for m in result["messages"] if m.type == "tool"])

    supervisor = create_agent(
        get_chat_model(
            settings=settings,
            responses=[tool_request("delegate", {"order_id": "BW1001"}), "Evidence received."],
        ),
        [delegate],
        middleware=[
            ModelCallLimitMiddleware(run_limit=3, exit_behavior="end"),
            ToolCallLimitMiddleware(run_limit=1, exit_behavior="end"),
        ],
    )
    # Use a fresh child for each measurement: finite fixtures do not rewind themselves.
    for name, agent in [
        ("single", specialist(settings, "C001", "logistics", "BW1001", "BW-001")),
        ("delegated", supervisor),
    ]:
        meter = Meter()
        start = perf_counter()
        result = agent.invoke(
            {"messages": [{"role": "user", "content": "Check shipping BW1001"}]},
            {"callbacks": [meter], "recursion_limit": 30},
        )
        # KEY IDEA: character/4 is a rough proxy, not provider token billing or a quality score.
        print_text(
            str(
                {
                    "pattern": name,
                    "model_calls": meter.calls,
                    "input_token_proxy": (meter.characters + 3) // 4,
                    "elapsed_ms": round((perf_counter() - start) * 1000, 2),
                    "tool_results": sum(m.type == "tool" for m in result["messages"]),
                }
            )
        )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Route to billing, product RAG and logistics SQL specialists; supervise the combined answer

Predict the output before running. The function definitions below match `use_case/operations_team.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Route a customer request to three restricted specialists and synthesize their evidence.

Run: uv run --extra agents python modules/16_multi_agent/use_case/operations_team.py --offline
Expected: Billing, product RAG and logistics reports; no writes or cross-customer data.
Cost: Up to twelve short calls including product retrieval generation. Requires: agents
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.operations import build_operations
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    # KEY IDEA: the authenticated host fixes C001; workers receive minimum necessary IDs.
    result = build_operations(settings, "C001").invoke(
        {
            "question": "Check billing total, tent product specifications, and shipping.",
            "order_id": "BW1001",
            "sku": "BW-001",
            "reports": [],
        },
        {"max_concurrency": 2, "recursion_limit": 12},
    )
    print_text(str(sorted(result["reports"], key=lambda report: report["role"])))
    print_text(result["answer"])


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra agents pytest modules/16_multi_agent/tests` for the supplied solution checks.